In [1]:
from bs4 import BeautifulSoup
from src.ingest.ingest_filings import fetch_html, get_primary_doc_url, get_cik, get_recent_filings
import re

In [2]:
cik, company = get_cik('MSFT')
print(f"CIK: {cik}, Company name: {company}")

CIK: 0000789019, Company name: MICROSOFT CORP


In [3]:
filings = get_recent_filings(cik, '10-Q',3)
print(f"Filings: {filings}")

  Found 3 10-Q filings
Filings: [{'form_type': '10-Q', 'filing_date': '2026-04-29', 'accession': '0001193125-26-191507', 'primary_doc': 'msft-20260331.htm'}, {'form_type': '10-Q', 'filing_date': '2026-01-28', 'accession': '0001193125-26-027207', 'primary_doc': 'msft-20251231.htm'}, {'form_type': '10-Q', 'filing_date': '2025-10-29', 'accession': '0001193125-25-256321', 'primary_doc': 'msft-20250930.htm'}]


In [4]:
filing_10q_msft_01 = filings[0]
filing_10q_msft_02 = filings[1]
filing_10q_msft_03 = filings[2]

In [63]:
doc_url_01 = get_primary_doc_url(cik, filing_10q_msft_01['accession'], filing_10q_msft_01['primary_doc'])
doc_url_02 = get_primary_doc_url(cik, filing_10q_msft_02['accession'], filing_10q_msft_02['primary_doc'])
doc_url_03 = get_primary_doc_url(cik, filing_10q_msft_03['accession'], filing_10q_msft_03['primary_doc'])
doc_url_04 = 'https://www.sec.gov/Archives/edgar/data/320193/000032019326000020/aapl-20260627.htm'

Trying document URL: https://www.sec.gov/Archives/edgar/data/789019/000119312526191507/msft-20260331.htm
Trying document URL: https://www.sec.gov/Archives/edgar/data/789019/000119312526027207/msft-20251231.htm
Trying document URL: https://www.sec.gov/Archives/edgar/data/789019/000119312525256321/msft-20250930.htm


In [ ]:
html_01_msft = fetch_html(doc_url_01)
html_02_msft = fetch_html(doc_url_02)
html_03_msft = fetch_html(doc_url_03)

In [64]:
html_04_appl = fetch_html(doc_url_04)

In [7]:
print(html_01_msft[:10000])

<?xml version='1.0' encoding='ASCII'?>
<!-- DFIN New ActiveDisclosure (SM) Inline XBRL Document - http://www.dfinsolutions.com/ -->
<!-- Creation Date :2026-04-29T14:08:38.0133+00:00 -->
<!-- Copyright (c) 2026 Donnelley Financial Solutions, Inc. All Rights Reserved. -->
<html xmlns="http://www.w3.org/1999/xhtml" xmlns:xbrldi="http://xbrl.org/2006/xbrldi" xmlns:xbrldt="http://xbrl.org/2005/xbrldt" xmlns:xbrli="http://www.xbrl.org/2003/instance" xmlns:link="http://www.xbrl.org/2003/linkbase" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:ix="http://www.xbrl.org/2013/inlineXBRL" xmlns:nonNumeric="http://www.xbrl.org/dtr/type/non-numeric" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:us-gaap="http://fasb.org/us-gaap/2025" xmlns:dtr-types="http://www.xbrl.org/dtr/type/2024-01-31" xmlns:country="http://xbrl.sec.gov/country/2025" xmlns:ixt-sec="http://www.sec.gov/inlineXBRL/transformation/2015-08-31" xmlns:msft="http://www.microsoft.com/20260331" xmlns:ecd-sub="http://xbrl.se

In [84]:
soup_01 = BeautifulSoup(html_03_msft, "lxml")

/tmp/ipykernel_434921/4262278799.py:1: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup_01 = BeautifulSoup(html_03_msft, "lxml")


In [85]:
for tag in soup_01(["script", "style"]):
    tag.decompose()
for tag in soup_01.find_all(["ix:header", "header"]):
    tag.decompose()
for tag in soup_01.find_all("ix:hidden"):
    tag.decompose()
for tag in soup_01.find_all(style=True):
    if tag.decomposed:
        continue
    style = tag.get("style", "").lower().replace(" ", "")
    if "display:none" in style or "visibility:hidden" in style:
        tag.decompose()

In [86]:
print(soup_01)

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [87]:
full_text = soup_01.get_text(separator="\n")
lines = []
for line in full_text.splitlines():
    line = line.strip()
    if not line or (len(line) < 25 and not re.match(
                    r'(?:PART|Part)\s+[IVX]+|(?:ITEM|Item)\s+\d+', line)):
        continue
    if line.startswith("http"):                  continue
    if re.match(r'^[\w\-]+:[\w\-]+', line):     continue
    if line.count(":") > 5 and len(line) < 300: continue
    lines.append(line)

print(f"length of clean lines: {len(lines)}")

length of clean lines: 901


In [88]:
full_text = "\n".join(lines)
# Match both Part headers and Item headers
PART_PATTERN = re.compile(
    r'(?:^|\n)((?:PART|Part)\s+[IVX]+\.?\s*[\xa0\s]*\w*)',
    re.MULTILINE
)
ITEM_PATTERN = re.compile(
    r'(?:^|\n)((?:ITEM|Item)\s+\d+[A-Za-z]?\.?\s*(?:[\xa0\s]+\w+.*)?)',
    re.MULTILINE
)

# Find all markers
all_matches = []
for m in PART_PATTERN.finditer(full_text):
    all_matches.append(("part", m))
for m in ITEM_PATTERN.finditer(full_text):
    all_matches.append(("item", m))

print(f"length of matches: {len(all_matches)}")
print(f"all matches: {all_matches}")

length of matches: 138
all matches: [('part', <re.Match object; span=(2073, 2086), match='\nPART I.\nItem'>), ('part', <re.Match object; span=(2726, 2740), match='\nPART II.\nItem'>), ('part', <re.Match object; span=(2836, 2848), match='\nPART I\nItem'>), ('part', <re.Match object; span=(2850, 2866), match='\nPART I. FINANCI'>), ('part', <re.Match object; span=(3114, 3126), match='\nPART I\nItem'>), ('part', <re.Match object; span=(3392, 3404), match='\nPART I\nItem'>), ('part', <re.Match object; span=(4155, 4167), match='\nPART I\nItem'>), ('part', <re.Match object; span=(5043, 5055), match='\nPART I\nItem'>), ('part', <re.Match object; span=(5484, 5496), match='\nPART I\nItem'>), ('part', <re.Match object; span=(9426, 9438), match='\nPART I\nItem'>), ('part', <re.Match object; span=(13808, 13820), match='\nPART I\nItem'>), ('part', <re.Match object; span=(17883, 17895), match='\nPART I\nItem'>), ('part', <re.Match object; span=(19417, 19429), match='\nPART I\nItem'>), ('part', <re.Ma

In [89]:
real_matches = []
for i, (mtype, match) in enumerate(all_matches):
    next_pos    = all_matches[i+1][1].start() \
                   if i+1 < len(all_matches) else len(full_text)
    content_len = next_pos - match.start()
    if content_len > 200:
        real_matches.append((mtype, match))
    
print(f"  Real section headers: {len(real_matches)}")

  Real section headers: 118


In [90]:
sections      = []
current_part  = "part_i"

for i, (mtype, match) in enumerate(real_matches):
    header = match.group(1).replace("\xa0", " ").strip()
    if mtype == "part":
        part_match = re.match(
            r'(?:PART|Part)\s+([IVX]+)', header, re.IGNORECASE
        )
        if part_match:
            roman     = part_match.group(1).upper()
            roman_map = {"I": "i", "II": "ii", "III": "iii", "IV": "iv"}
            current_part = f"part_{roman_map.get(roman, roman.lower())}"
        label = current_part
    else:
        item_match = re.match(
                        r'(?:ITEM|Item)\s+(\d+[A-Za-z]?)\.?', header, re.IGNORECASE
                    )
        item_num = item_match.group(1).lower() if item_match else f"item_{i}"
        label    = f"{current_part}_item_{item_num}"
        
    content_start = match.end()
    content_end   = real_matches[i+1][1].start() \
                    if i+1 < len(real_matches) else len(full_text)
    section_text  = full_text[content_start:content_end].strip()
    
    if len(section_text) < 100:
                continue
    
    xbrl_hits = section_text.count("us-gaap") + section_text.count("fasb.org")
    if xbrl_hits > 10 and len(section_text.split()) < 500:
        continue

    sections.append({
        "section": label,
        "title":   header[:60],
        "text":    section_text
    })

    # print(f"  [{label}] {header[:40]:<40} {len(section_text):>8,} chars")
    
print(f"section: {sections[:2]}")
        
        

section: [{'section': 'part_i', 'title': 'PART I.\nItem', 'text': '1.\nIncome Statements for the Three Months Ended September 30, 2025 and 2024\nComprehensive Income Statements for the Three Months Ended September 30, 2025 and 2024\nBalance Sheets as of September 30, 2025 and June 30, 2025\nCash Flows Statements for the Three Months Ended September 30, 2025 and 2024\nStockholders’ Equity Statements for the Three Months Ended September 30, 2025 and 2024\nNotes to Financial Statements\nReport of Independent Registered Public Accounting Firm\nItem 2.\nManagement’s Discussion and Analysis of Financial Condition and Results of Operations\nItem 3.\nQuantitative and Qualitative Disclosures About Market Risk\nItem 4.\nPART II.\nItem 1.\nItem 1A.\nItem 2.\nUnregistered Sales of Equity Securities and Use of Proceeds\nItem 5.\nItem 6.\nPART I\nItem 1'}, {'section': 'part_i', 'title': 'PART I. FINANCI', 'text': 'ITEM 1. FINANCI\n(In millions, except per share amounts) (Unaudited)\nThree Months End

In [31]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
CHUNK_SIZE  = 500   # tokens
CHUNK_OVERLAP = 50  # tokens

In [91]:
tokenizer = tiktoken.get_encoding("cl100k_base")
splitter  = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=lambda t: len(tokenizer.encode(t)),
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
for section in sections:
    for i, text in enumerate(splitter.split_text(section["text"])):
        chunks.append({
            "ticker":        'MSFT',
            "filing_date":   filing_10q_msft_01['filing_date'],
            "form_type":     filing_10q_msft_01['form_type'],
            "section":       section["section"],
            "section_title": section.get("title", ""),
            "chunk_index":   i,
            "text":          text,
            "token_count":   len(tokenizer.encode(text)),
        })
    
print(f"chunks: ${chunks}")

chunks: $[{'ticker': 'MSFT', 'filing_date': '2026-04-29', 'form_type': '10-Q', 'section': 'part_i', 'section_title': 'PART I.\nItem', 'chunk_index': 0, 'text': '1.\nIncome Statements for the Three Months Ended September 30, 2025 and 2024\nComprehensive Income Statements for the Three Months Ended September 30, 2025 and 2024\nBalance Sheets as of September 30, 2025 and June 30, 2025\nCash Flows Statements for the Three Months Ended September 30, 2025 and 2024\nStockholders’ Equity Statements for the Three Months Ended September 30, 2025 and 2024\nNotes to Financial Statements\nReport of Independent Registered Public Accounting Firm\nItem 2.\nManagement’s Discussion and Analysis of Financial Condition and Results of Operations\nItem 3.\nQuantitative and Qualitative Disclosures About Market Risk\nItem 4.\nPART II.\nItem 1.\nItem 1A.\nItem 2.\nUnregistered Sales of Equity Securities and Use of Proceeds\nItem 5.\nItem 6.\nPART I\nItem 1', 'token_count': 196}, {'ticker': 'MSFT', 'filing_date

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("BAAI/bge-base-en-v1.5")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
texts      = [c["text"] for c in chunks]
print(texts)
embeddings = model.encode(texts, batch_size=32,
                              show_progress_bar=True,
                              normalize_embeddings=True)


['1.\nIncome Statements for the Three and Six Months Ended December 31, 2025 and 2024\nComprehensive Income Statements for the Three and Six Months Ended December 31, 2025 and 2024\nBalance Sheets as of December 31, 2025 and June 30, 2025\nCash Flows Statements for the Three and Six Months Ended December 31, 2025 and 2024\nStockholders’ Equity Statements for the Three and Six Months Ended December 31, 2025 and 2024\nNotes to Financial Statements\nReport of Independent Registered Public Accounting Firm\nItem 2.\nManagement’s Discussion and Analysis of Financial Condition and Results of Operations\nItem 3.\nQuantitative and Qualitative Disclosures About Market Risk\nItem 4.\nPART II.\nItem 1.\nItem 1A.\nItem 2.\nUnregistered Sales of Equity Securities and Use of Proceeds\nItem 5.\nItem 6.\nPART I\nItem 1', 'ITEM 1. FINANCI\n(In millions, except per share amounts) (Unaudited)\nGeneral and administrative\nOther income (expense), net\nIncome before income taxes\nProvision for income taxes\n

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [81]:
for chunk, emb in zip(chunks,embeddings):
    chunk['embedding'] = emb.tolist()

rows = [
    (c["ticker"], c["filing_date"], c["form_type"], c["section"], c["section_title"],
             c["chunk_index"], c["text"], c["token_count"], c["embedding"])
            for c in chunks
]

print(len(rows))

204


In [52]:
from dotenv import load_dotenv
import os
load_dotenv()

CONN_STR    = os.getenv("DATABASE_URL")

In [82]:
import psycopg2
from psycopg2.extras import execute_values

conn = psycopg2.connect(CONN_STR)
cur = conn.cursor()

execute_values(
    cur,
    """INSERT INTO filing_chunks
        (ticker, filing_date, form_type, section, section_title,
        chunk_index, text, token_count, embedding)
        VALUES %s""",
    rows,
    template="(%s,%s,%s,%s,%s,%s,%s,%s,%s::vector)"
)

In [83]:
conn.commit()
print(f"  Stored {len(rows)} chunks")
cur.close()
conn.close()

  Stored 204 chunks
